In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('tmdb_movies.db')

queries = {
    "Q1: Action movies released after 2015": """
        SELECT m.title, m.release_date
        FROM movies m
        JOIN movie_genres mg ON m.movie_id = mg.movie_id
        JOIN genres g ON mg.genre_id = g.genre_id
        WHERE g.genre_name = 'Action' AND m.release_date > '2015-12-31'
        ORDER BY m.release_date DESC;
    """,
    "Q2: Top 10 highest-grossing movies with genres": """
        SELECT m.title, m.revenue, GROUP_CONCAT(g.genre_name, ', ') AS genres
        FROM movies m
        LEFT JOIN movie_genres mg ON m.movie_id = mg.movie_id
        LEFT JOIN genres g ON mg.genre_id = g.genre_id
        GROUP BY m.movie_id, m.title, m.revenue
        ORDER BY m.revenue DESC
        LIMIT 10;
    """,
    "Q3: Average budget and revenue by genre": """
        SELECT g.genre_name,
               ROUND(AVG(NULLIF(m.budget, 0)), 2) AS avg_budget,
               ROUND(AVG(NULLIF(m.revenue, 0)), 2) AS avg_revenue
        FROM genres g
        JOIN movie_genres mg ON g.genre_id = mg.genre_id
        JOIN movies m ON mg.movie_id = m.movie_id
        GROUP BY g.genre_name
        ORDER BY avg_revenue DESC;
    """,
    "Q4: Top 10 actors by movie appearances": """
        SELECT actor_name, COUNT(DISTINCT movie_id) AS movie_count
        FROM cast
        GROUP BY person_id, actor_name
        ORDER BY movie_count DESC
        LIMIT 10;
    """,
    "Q5: Directors with more than 3 movies": """
        SELECT person_name AS director, COUNT(DISTINCT movie_id) AS movie_count
        FROM crew
        WHERE job = 'Director'
        GROUP BY person_id, person_name
        HAVING COUNT(DISTINCT movie_id) > 3
        ORDER BY movie_count DESC;
    """,
    "Q6: Top 10 most frequently used keywords": """
        SELECT keyword_name, COUNT(DISTINCT movie_id) AS usage_count
        FROM movie_keywords
        GROUP BY keyword_id, keyword_name
        ORDER BY usage_count DESC
        LIMIT 10;
    """,
    "Q7: Above-average budget but below-average revenue": """
        SELECT title, budget, revenue
        FROM movies
        WHERE budget > (SELECT AVG(budget) FROM movies WHERE budget > 0)
          AND revenue > 0
          AND revenue < (SELECT AVG(revenue) FROM movies WHERE revenue > 0)
        ORDER BY budget DESC;
    """,
    "Q8: Actors who appeared in a Christopher Nolan movie": """
        SELECT DISTINCT c.actor_name, m.title AS movie_title
        FROM cast c
        JOIN crew cr ON c.movie_id = cr.movie_id
        JOIN movies m ON c.movie_id = m.movie_id
        WHERE cr.job = 'Director' AND cr.person_name = 'Christopher Nolan'
        ORDER BY m.title, c.actor_name;
    """,
    "Q9: Genre with highest average rating (min 20 movies)": """
        SELECT g.genre_name,
               COUNT(m.movie_id) AS total_movies,
               ROUND(AVG(m.vote_average), 2) AS avg_rating
        FROM genres g
        JOIN movie_genres mg ON g.genre_id = mg.genre_id
        JOIN movies m ON mg.movie_id = m.movie_id
        GROUP BY g.genre_id, g.genre_name
        HAVING COUNT(m.movie_id) >= 20
        ORDER BY avg_rating DESC
        LIMIT 1;
    """,
    "Q10: Top 10 movies with the highest number of keywords": """
        SELECT m.title, COUNT(mk.keyword_id) AS keyword_count
        FROM movies m
        JOIN movie_keywords mk ON m.movie_id = mk.movie_id
        GROUP BY m.movie_id, m.title
        ORDER BY keyword_count DESC
        LIMIT 10;
    """
}

for title, q in queries.items():
    print(f"\n--- {title} ---")
    display(pd.read_sql_query(q, conn))

conn.close()
